# *SQL*

El coronavirus tomó al mundo entero por sorpresa, cambiando la rutina diaria de todos y todas. Los habitantes de las ciudades ya no pasaban su tiempo libre fuera, yendo a cafés y centros comerciales; sino que más gente se quedaba en casa, leyendo libros. Eso atrajo la atención de las startups (empresas emergentes) que se apresuraron a desarrollar nuevas aplicaciones para los amantes de los libros.

Te han dado una base de datos de uno de los servicios que compiten en este mercado. Contiene datos sobre libros, editoriales, autores y calificaciones de clientes y reseñas de libros. Esta información se utilizará para generar una propuesta de valor para un nuevo producto.

## 1) Objetivos del estudio

Queremos usar la base para entender el catálogo y el comportamiento de usuarios, para apoyar una propuesta de valor de una app de lectura:

	•Medir crecimiento del catálogo moderno (después del 2000).
	•Evaluar qué libros generan más interacción (reseñas + rating promedio).
	•Detectar la editorial más fuerte en “libros reales” (más de 50 páginas).
	•Identificar al autor mejor valorado con evidencia suficiente (≥ 50 calificaciones).
	•Medir el hábito de reseñar entre usuarios muy activos (califican > 50 libros).

## Conectarse a la bases de datos

In [14]:
import pandas as pd
from sqlalchemy import create_engine

db_config = {
    'user': 'practicum_student',
    'pwd': 's65BlTKV3faNIGhmvJVzOqhs',
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': 6432,
    'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

La conexión se almacena en la variable engine. Ejecutamos una consulta SQL utilizando pandas:

In [15]:
pd.read_sql("SELECT * FROM books LIMIT 5;", engine)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


Con este código se hace la consulta y vemos que ya estamos conectados.

### Encuentra el número de libros publicados después del 1 de enero de 2000.

In [16]:
query_1 = """
SELECT COUNT(*) AS books_after_2000
FROM books
WHERE publication_date > '2000-01-01';
"""
pd.read_sql(query_1, engine)

,books_after_2000
0,819


Número de libros publicados después del 1 de enero de 2000

Para identificar qué tan moderno es el catálogo disponible en la base de datos, se calculó el número de libros publicados después del 1 de enero del año 2000.

Se identificaron 819 libros publicados después del año 2000.

El catálogo contiene una cantidad considerable de libros modernos, lo que sugiere que la plataforma incluye una oferta relevante de títulos contemporáneos. Esto puede ser atractivo para usuarios que buscan lecturas recientes y tendencias actuales.

## Encuentra el número de reseñas de usuarios y la calificación promedio para cada libro.

In [17]:
query_2 = """
SELECT
b.book_id,
b.title,
COALESCE(rvw.review_count, 0) AS review_count,
ROUND(COALESCE(rt.avg_rating, 0), 2) AS avg_rating
FROM books b
LEFT JOIN (
SELECT book_id, COUNT(*) AS review_count
FROM reviews
GROUP BY book_id
) rvw ON b.book_id = rvw.book_id
LEFT JOIN (
SELECT book_id, AVG(rating) AS avg_rating
FROM ratings
GROUP BY book_id
) rt ON b.book_id = rt.book_id
ORDER BY review_count DESC, avg_rating DESC;
"""
result_2 = pd.read_sql(query_2, engine)
result_2.head(10)

,book_id,title,review_count,avg_rating
0,948,Twilight (Twilight #1),7,3.66
1,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.41
2,299,Harry Potter and the Chamber of Secrets (Harry...,6,4.29
3,656,The Book Thief,6,4.26
4,734,The Glass Castle,6,4.21
5,497,Outlander (Outlander #1),6,4.13
6,750,The Hobbit or There and Back Again,6,4.13
7,695,The Curious Incident of the Dog in the Night-Time,6,4.08
8,779,The Lightning Thief (Percy Jackson and the Oly...,6,4.08
9,963,Water for Elephants,6,3.98


**Número de reseñas y calificación promedio por libro**

Se calculó el número de reseñas de texto y la calificación promedio para cada libro con el objetivo de evaluar el nivel de interacción y la percepción de calidad por parte de los usuarios.

 Resultados principales

 Los libros con mayor número de reseñas fueron:
	•Twilight (7 reseñas, rating promedio 3.66)
	•Harry Potter and the Prisoner of Azkaban (6 reseñas, rating 4.41)
	•Harry Potter and the Chamber of Secrets (6 reseñas, rating 4.29)
	•The Book Thief (6 reseñas, rating 4.26)

**Conclusión**

El número máximo de reseñas por libro es relativamente bajo (7), lo que indica un volumen moderado de interacción dentro del dataset.

Un mayor número de reseñas no necesariamente implica una mejor calificación, como se observa en el caso de Twilight.



## Identifica la editorial que ha publicado el mayor número de libros con más de 50 páginas (esto te ayudará a excluir folletos y publicaciones similares de tu análisis).

In [18]:
query_3 = """
SELECT
p.publisher,
COUNT(b.book_id) AS books_over_50_pages
FROM publishers p
JOIN books b 
ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY books_over_50_pages DESC
LIMIT 1;
"""
pd.read_sql(query_3, engine)

,publisher,books_over_50_pages
0,Penguin Books,42


**Editorial con mayor número de libros (más de 50 páginas)**

La editorial con mayor número de libros extensos es Penguin Books, con un total de 42 libros.

Penguin Books lidera el catálogo en términos de volumen de publicaciones completas.
Este resultado sugiere una fuerte presencia editorial dentro de la base de datos y la posiciona como un actor clave para posibles estrategias de contenido o alianzas comerciales en una nueva plataforma de lectura.

## Identifica al autor que tiene la más alta calificación promedio del libro: mira solo los libros con al menos 50 calificaciones.

In [19]:
query_4 = """
WITH books_50_ratings AS (
SELECT book_id
FROM ratings
GROUP BY book_id
HAVING COUNT(*) >= 50
),
author_avg AS (
SELECT
b.author_id,
AVG(r.rating) AS avg_rating
FROM books b
JOIN books_50_ratings br ON b.book_id = br.book_id
JOIN ratings r ON r.book_id = b.book_id
GROUP BY b.author_id
)
SELECT
a.author,
ROUND(aa.avg_rating, 2) AS avg_rating
FROM author_avg aa
JOIN authors a ON a.author_id = aa.author_id
ORDER BY aa.avg_rating DESC
LIMIT 1;
"""
pd.read_sql(query_4, engine)

,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.29


**Autor con mayor calificación promedio (libros con ≥ 50 calificaciones)**

Para garantizar resultados confiables, se analizaron únicamente libros que cuentan con al menos 50 calificaciones. Esto evita que el promedio esté influenciado por una muestra pequeña de usuarios.

El autor con mayor calificación promedio es:

J.K. Rowling/Mary GrandPré, con un promedio de 4.29.

Este resultado indica una alta satisfacción de los lectores hacia las obras de estos autores dentro del dataset analizado.

Al estar basado en libros con un número considerable de calificaciones, el promedio obtenido es estadísticamente sólido y representa una preferencia consistente por parte de los usuarios.

## Encuentra el número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros.

In [20]:
query_5 = """
WITH power_users AS (
SELECT username
FROM ratings
GROUP BY username
HAVING COUNT(*) > 50
),
reviews_per_user AS (
SELECT
pu.username,
COUNT(rw.review_id) AS review_count
FROM power_users pu
LEFT JOIN reviews rw 
ON rw.username = pu.username
GROUP BY pu.username
)
SELECT ROUND(AVG(review_count), 2) AS avg_reviews_among_power_users
FROM reviews_per_user;
"""
pd.read_sql(query_5, engine)

,avg_reviews_among_power_users
0,24.33


**Promedio de reseñas entre usuarios que calificaron más de 50 libros**

Para analizar el comportamiento de los usuarios más activos, se identificaron aquellos que han calificado más de 50 libros. Posteriormente, se calculó el número promedio de reseñas de texto escritas por este grupo.

La consulta permitió determinar el nivel de participación real de estos usuarios dentro de la plataforma.

El promedio de reseñas de texto entre estos usuarios es de 24.33 reseñas.

Los usuarios altamente activos no solo califican libros, sino que también contribuyen con una cantidad significativa de reseñas escritas.

Esto indica un nivel importante de compromiso dentro del grupo más participativo de la plataforma. Estos usuarios pueden ser clave para fomentar la comunidad, generar contenido valioso y fortalecer la experiencia social en una aplicación de lectura digital.

# **Conclusiones generales**

Se realizó un análisis de la base de datos con el objetivo de evaluar el catálogo, la calidad de los autores y el nivel de participación de los usuarios.

Los principales hallazgos fueron:

    •Se identificaron 819 libros publicados después del año 2000, lo que indica una oferta considerable de contenido moderno.
    •Los libros más populares presentan calificaciones promedio superiores a 4.0, lo que refleja alta satisfacción de los lectores.
    •Penguin Books es la editorial con mayor número de libros extensos (42 títulos con más de 50 páginas).
    •J.K. Rowling/Mary GrandPré es el autor con la calificación promedio más alta (4.29) considerando libros con al menos 50 calificaciones.
    •Los usuarios que califican más de 50 libros escriben en promedio 24.33 reseñas, lo que demuestra un nivel significativo de compromiso.

En general, el análisis muestra un catálogo sólido, autores bien valorados y una comunidad activa, lo cual representa una base favorable para el desarrollo de una plataforma digital de lectura.
    